# TACO EDA

[TACO](http://tacodataset.org) (Trash Annotations in Context) is a COCO-format detection/segmentation dataset of litter photographed in the wild. We use it for **classification** by cropping every annotated object (see `data/taco_dataset.py`), so this notebook looks at both the photos and the object crops.

**Getting the data**
```shell
git clone https://github.com/pedropro/TACO.git
cd TACO
pip install -r requirements.txt
python download.py   # images are pulled from Flickr into data/batch_*/
```
Then point `TACO_ROOT` in `data/constants.py` at `TACO/data`.

Run this notebook from `data/eda/taco/`.

In [ ]:
import sys
sys.path.insert(0, "../../..")  # repo root

import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
from torchvision import transforms

from data.constants import TACO_ROOT, SEED
from data.taco_dataset import TACODataset

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
random.seed(SEED)

DATA_PATH = Path(TACO_ROOT)
MAX_IMAGES_FOR_PIXEL_STATS = 300  # opening full-res photos is slow

# **Data Overview**

In [ ]:
with open(DATA_PATH / "annotations.json") as f:
    coco = json.load(f)

images_df = pd.DataFrame(coco["images"])
cats_df = pd.DataFrame(coco["categories"]).rename(columns={"id": "category_id", "name": "category"})
anns_df = pd.DataFrame(coco["annotations"]).merge(cats_df[["category_id", "category", "supercategory"]], on="category_id")
anns_df = anns_df.merge(images_df[["id", "file_name", "width", "height"]].rename(columns={"id": "image_id"}), on="image_id")

print(f"Images:          {len(images_df):,}")
print(f"Annotations:     {len(anns_df):,}")
print(f"Categories:      {cats_df['category'].nunique()}")
print(f"Supercategories: {cats_df['supercategory'].nunique()}")
print(f"Batches:         {images_df['file_name'].str.split('/').str[0].nunique()}")
images_df.head()

## Sanity checks
Images are downloaded from Flickr, so some may be missing or broken. We also check EXIF orientation: TACO annotations are in the EXIF-rotated frame, so if the raw file size disagrees with the annotated `width`/`height` the loader **must** call `ImageOps.exif_transpose` before cropping.

In [ ]:
missing, corrupted, exif_rotated, size_mismatch = [], [], [], []

for row in images_df.itertuples():
    path = DATA_PATH / row.file_name
    if not path.is_file():
        missing.append(row.file_name)
        continue
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            raw_size = img.size
            upright_size = ImageOps.exif_transpose(img).size
    except Exception:
        corrupted.append(row.file_name)
        continue
    if raw_size != upright_size:
        exif_rotated.append(row.file_name)
    if upright_size != (row.width, row.height):
        size_mismatch.append(row.file_name)

print(f"Missing files:                          {len(missing)}")
print(f"Corrupted files:                        {len(corrupted)}")
print(f"EXIF-rotated files:                     {len(exif_rotated)}")
print(f"Upright size != annotated size:         {len(size_mismatch)}")
print(f"Images with no annotations:             {len(set(images_df['id']) - set(anns_df['image_id']))}")

# **Class Distribution**

In [ ]:
super_counts = anns_df["supercategory"].value_counts()
cat_counts = anns_df["category"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
super_counts.sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Objects per Supercategory", fontweight="bold")
axes[0].set_xscale("log")
cat_counts.sort_values().plot.barh(ax=axes[1], color="seagreen", fontsize=6)
axes[1].set_title("Objects per Category", fontweight="bold")
axes[1].set_xscale("log")
plt.tight_layout()
plt.show()

print(f"Supercategory imbalance (max/min): {super_counts.max() / super_counts.min():.0f}x")
print(f"Category imbalance (max/min):      {cat_counts.max() / cat_counts.min():.0f}x")
print(f"Supercategories with < 20 objects: {list(super_counts[super_counts < 20].index)}")

In [ ]:
# How much of the data the top classes cover
cum = (super_counts.cumsum() / super_counts.sum()).round(3)
pd.DataFrame({"count": super_counts, "cumulative_share": cum})

# **Objects per Image**

In [ ]:
per_image = anns_df.groupby("image_id").agg(n_objects=("id", "count"), n_supercats=("supercategory", "nunique"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
per_image["n_objects"].plot.hist(bins=range(1, per_image["n_objects"].max() + 2), ax=axes[0], color="steelblue")
axes[0].set_title("Annotated Objects per Image", fontweight="bold")
axes[0].set_xlabel("objects")
per_image["n_supercats"].plot.hist(bins=range(1, per_image["n_supercats"].max() + 2), ax=axes[1], color="seagreen")
axes[1].set_title("Distinct Supercategories per Image", fontweight="bold")
axes[1].set_xlabel("supercategories")
plt.tight_layout()
plt.show()

per_image.describe().round(2)

# **Image Dimensions**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(images_df["width"], images_df["height"], alpha=0.3, s=10)
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Height (px)")
axes[0].set_title("Resolution Scatter", fontweight="bold")
(images_df["width"] / images_df["height"]).plot.hist(bins=40, ax=axes[1], color="steelblue")
axes[1].axvline(1.0, color="red", linestyle="--")
axes[1].set_title("Aspect Ratio (w/h)", fontweight="bold")
plt.tight_layout()
plt.show()

images_df[["width", "height"]].describe().round(0)

# **Bounding Boxes (our classification crops)**
Crop size matters more than photo size: every crop gets resized to `IMG_DIM` in `data/constants.py`, so tiny crops get upsampled heavily.

In [ ]:
anns_df["bbox_w"] = anns_df["bbox"].str[2]
anns_df["bbox_h"] = anns_df["bbox"].str[3]
anns_df["min_side"] = anns_df[["bbox_w", "bbox_h"]].min(axis=1)
anns_df["rel_area"] = (anns_df["bbox_w"] * anns_df["bbox_h"]) / (anns_df["width"] * anns_df["height"])
anns_df["bbox_aspect"] = anns_df["bbox_w"] / anns_df["bbox_h"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
anns_df["min_side"].plot.hist(bins=np.logspace(0, 4, 50), ax=axes[0], color="steelblue")
axes[0].set_xscale("log")
axes[0].set_title("Crop Shorter Side (px)", fontweight="bold")
anns_df["rel_area"].plot.hist(bins=np.logspace(-6, 0, 50), ax=axes[1], color="seagreen")
axes[1].set_xscale("log")
axes[1].set_title("Bbox Area / Image Area", fontweight="bold")
anns_df["bbox_aspect"].plot.hist(bins=np.logspace(-1.5, 1.5, 50), ax=axes[2], color="tomato")
axes[2].set_xscale("log")
axes[2].set_title("Bbox Aspect Ratio (w/h)", fontweight="bold")
plt.tight_layout()
plt.show()

for t in (16, 32, 64, 224):
    print(f"Crops with shorter side < {t:>3}px: {(anns_df['min_side'] < t).mean():.1%}")

In [ ]:
# Crop size per supercategory: are some classes systematically tiny (e.g. cigarettes)?
order = anns_df.groupby("supercategory")["min_side"].median().sort_values().index
fig, ax = plt.subplots(figsize=(14, 5))
ax.boxplot([anns_df.loc[anns_df["supercategory"] == s, "min_side"] for s in order], showfliers=False)
ax.set_xticklabels(order, rotation=75)
ax.set_yscale("log")
ax.set_ylabel("Crop shorter side (px)")
ax.set_title("Crop Size per Supercategory", fontweight="bold")
plt.tight_layout()
plt.show()

# **Example Images with Annotations**

In [ ]:
def show_annotated(image_ids, n_cols=3):
    n_rows = int(np.ceil(len(image_ids) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5), squeeze=False)
    for ax, image_id in zip(axes.flat, image_ids):
        row = images_df[images_df["id"] == image_id].iloc[0]
        img = ImageOps.exif_transpose(Image.open(DATA_PATH / row["file_name"])).convert("RGB")
        ax.imshow(img)
        for a in anns_df[anns_df["image_id"] == image_id].itertuples():
            x, y, w, h = a.bbox
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color="red", linewidth=2))
            ax.text(x, y, a.supercategory, color="white", fontsize=7, backgroundcolor="red")
        ax.set_title(row["file_name"], fontsize=8)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_annotated(random.sample(list(per_image.index), min(6, len(per_image))))

# **Example Crops per Supercategory**
What the classifier actually sees, loaded through `TACODataset`.

In [ ]:
crop_ds = TACODataset(DATA_PATH, split="all", transform=transforms.Resize((128, 128)))
by_class = {}
for i, (_, _, label) in enumerate(crop_ds.samples):
    by_class.setdefault(label, []).append(i)

N_PER_CLASS = 6
fig, axes = plt.subplots(len(crop_ds.classes), N_PER_CLASS,
                         figsize=(N_PER_CLASS * 1.6, len(crop_ds.classes) * 1.6), squeeze=False)
for label, cls in enumerate(crop_ds.classes):
    idxs = random.sample(by_class.get(label, []), min(N_PER_CLASS, len(by_class.get(label, []))))
    for col in range(N_PER_CLASS):
        ax = axes[label, col]
        ax.axis("off")
        if col < len(idxs):
            ax.imshow(crop_ds[idxs[col]][0])
    axes[label, 0].text(-10, 64, cls, ha="right", va="center", fontsize=8)
plt.tight_layout()
plt.show()

# **Pixel Intensity**
Mean RGB and brightness of crops per supercategory (sampled).

In [ ]:
sample_idx = random.sample(range(len(crop_ds)), min(MAX_IMAGES_FOR_PIXEL_STATS, len(crop_ds)))
records = []
for i in sample_idx:
    img, label = crop_ds[i]
    arr = np.asarray(img, dtype=np.float32)
    records.append({"supercategory": crop_ds.classes[label], "R": arr[..., 0].mean(),
                    "G": arr[..., 1].mean(), "B": arr[..., 2].mean(), "brightness": arr.mean()})
pixel_df = pd.DataFrame(records).groupby("supercategory").mean().sort_values("brightness")

pixel_df[["R", "G", "B"]].plot.bar(figsize=(14, 4), color=["tomato", "mediumseagreen", "steelblue"])
plt.title("Mean RGB per Supercategory (sampled crops)", fontweight="bold")
plt.ylabel("Mean pixel value (0-255)")
plt.tight_layout()
plt.show()

# **Train / Val / Test Split**
TACO has no official split; `TACODataset` splits by image id (seeded) so crops from one photo never land in two splits. Check every class shows up in val/test.

In [ ]:
split_counts = {}
for split in ("train", "val", "test"):
    ds = TACODataset(DATA_PATH, split=split, transform=lambda x: x)
    split_counts[split] = Counter(ds.classes[label] for _, _, label in ds.samples)
split_df = pd.DataFrame(split_counts).fillna(0).astype(int).sort_values("train", ascending=False)
split_df["total"] = split_df.sum(axis=1)

print(split_df.sum())
print("Classes missing from val: ", list(split_df.index[split_df["val"] == 0]))
print("Classes missing from test:", list(split_df.index[split_df["test"] == 0]))
split_df

# **Observations**
_Fill in after running on the full dataset._

- **Class imbalance**: ...
- **Small crops**: ...
- **EXIF orientation**: ...
- **Split coverage**: ...
- **Next steps** (e.g. merge rare supercategories into `Other`, drop crops below N px, class-weighted loss): ...